# 🎙️ Disfluency Detection & Audio Clipping Pipeline

This notebook reads a Microsoft Excel dataset, fixes broken URLs, downloads transcription JSONs, detects disfluencies in Hindi/English text, clips disfluent audio segments, and produces a structured CSV.

---

### Disfluency Categories
| Category | Examples |
|---|---|
| **Fillers** | `uh`, `umm`, `मतलब`, `तो`, `आ…` |
| **Repetitions** | `मैं मैं`, `वो वो` |
| **False Starts** | `मैं कल… मतलब परसों गया` |
| **Prolongations** | `सोऽऽऽ`, `आआआ`, `soooo` |
| **Hesitations** | Short/abnormal duration segments |

## 1. Install Dependencies

In [ ]:
!pip install pandas openpyxl requests pydub librosa soundfile tqdm -q

## 2. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import json
import unicodedata
import warnings
import random
import logging
from pathlib import Path
from collections import Counter, defaultdict

import requests
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')


EXCEL_FILE = "dataset.xlsx"

OUTPUT_DIR   = "output"
CLIPS_DIR    = os.path.join(OUTPUT_DIR, "clips")
CACHE_DIR    = os.path.join(OUTPUT_DIR, ".cache")
OUTPUT_CSV   = os.path.join(OUTPUT_DIR, "disfluency_results.csv")
ERROR_LOG    = os.path.join(OUTPUT_DIR, "errors.log")
SAMPLE_RATE  = 16000  

for d in [OUTPUT_DIR, CLIPS_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)


logging.basicConfig(
    filename=ERROR_LOG,
    level=logging.WARNING,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print(f"📁 Excel file  : {EXCEL_FILE}")
print(f"📂 Output dir  : {OUTPUT_DIR}")
print(f"🔊 Sample rate : {SAMPLE_RATE} Hz")

## 3. Load Excel Dataset & Fix Broken URLs

Fixes the broken JoshTalks URL pattern:  
`https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/{rec_id}/{file}`  
→ `https://storage.googleapis.com/upload_goai/{rec_id}/{file}`

In [ ]:

OLD_URL_PATTERN = r'https?://storage\.googleapis\.com/joshtalks-data-collection/hq_data/hi/'
NEW_URL_BASE    = 'https://storage.googleapis.com/upload_goai/'

def fix_broken_url(url):
    """Fix the broken JoshTalks URL to the correct upload_goai path."""
    if pd.isna(url) or not isinstance(url, str):
        return url
    return re.sub(OLD_URL_PATTERN, NEW_URL_BASE, url)


df = pd.read_excel(EXCEL_FILE, engine='openpyxl')
print(f"✅ Loaded {len(df)} rows from '{EXCEL_FILE}'")
print(f"📋 Columns: {list(df.columns)}")
print()
df.head(3)

In [ ]:

def detect_column(df, keywords, label):
    """Auto-detect a column by matching keywords in column names."""
    for col in df.columns:
        col_lower = str(col).lower().replace(' ', '_').replace('-', '_')
        if any(kw in col_lower for kw in keywords):
            return col
    
    print(f"⚠️  Could not auto-detect '{label}' column.")
    print(f"    Available columns: {list(df.columns)}")
    return None


id_col    = detect_column(df, ['recording_id', 'rec_id', 'id', 'recording'], 'recording_id')
trans_col = detect_column(df, ['transcription', 'transcript', 'trans_url', 'json_url'], 'transcription_url')
audio_col = detect_column(df, ['audio_url', 'audio', 'wav_url', 'mp3_url', 'sound'], 'audio_url')

print(f"🔑 Recording ID column  : {id_col}")
print(f"📝 Transcription URL col: {trans_col}")
print(f"🔊 Audio URL column     : {audio_col}")


url_cols_fixed = []
for col in df.columns:
    if df[col].dtype == object:  
        sample = df[col].dropna().head(5).astype(str)
        if sample.str.contains('googleapis', case=False).any():
            df[col] = df[col].apply(fix_broken_url)
            url_cols_fixed.append(col)

print(f"\n🔧 Fixed URLs in columns: {url_cols_fixed}")
if url_cols_fixed:
    print(f"   Sample fixed URL: {df[url_cols_fixed[0]].dropna().iloc[0]}")

## 4. Text Preprocessing & Normalization

In [ ]:
def preprocess_text(text):
    """
    Preprocess transcription text:
    - Unicode NFC normalization
    - Collapse extra whitespace
    - Remove punctuation noise (keep Devanagari, alphanumeric)
    - Lowercase Latin characters only (preserve Devanagari case)
    """
    if not text or not isinstance(text, str):
        return ""
    
    
    text = unicodedata.normalize('NFC', text)
    
    
    result = []
    for ch in text:
        if 'A' <= ch <= 'Z':
            result.append(ch.lower())
        else:
            result.append(ch)
    text = ''.join(result)
    
    
    text = re.sub(r'[^\u0900-\u097F\u0980-\u09FFa-z0-9\s\.…ऽ]', ' ', text)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

test_texts = [
    "मैं   कल...  मतलब  परसों गया!!!",
    "UH, so I was LIKE, you know...",
    "आआआ वो   वो मतलब",
]
print("─── Preprocessing Demo ───")
for t in test_texts:
    print(f"  IN:  {t}")
    print(f"  OUT: {preprocess_text(t)}")
    print()

## 5. Disfluency Detection Engine

Five detection categories:
1. **Fillers** — Dictionary lookup (Hindi + English)
2. **Repetitions** — Consecutive duplicate tokens (`word[i] == word[i+1]`)
3. **False Starts** — Correction markers between different words
4. **Prolongations** — ≥3 repeated characters
5. **Hesitations** — Duration-based anomalies

In [ ]:

HINDI_FILLERS = {
    'मतलब', 'तो', 'अच्छा', 'हां', 'हाँ', 'ना', 'बस', 'ये',
    'वो', 'कि', 'जैसे', 'अरे', 'ओह', 'हम्म', 'आ', 'ऐसा',
    'बोलो', 'देखो', 'सुनो', 'यार', 'भई', 'अब', 'फिर',
    'वैसे', 'खैर', 'चलो', 'हूं', 'हूँ', 'क्या', 'कुछ',
    'वगैरह', 'इत्यादि', 'बोले', 'समझे', 'पता',
}

ENGLISH_FILLERS = {
    'uh', 'um', 'umm', 'uhh', 'hmm', 'hm', 'hmm',
    'like', 'you know', 'so', 'well', 'actually',
    'basically', 'literally', 'right', 'okay', 'ok',
    'i mean', 'kind of', 'sort of', 'er', 'ah', 'oh',
}

ALL_FILLERS = HINDI_FILLERS | ENGLISH_FILLERS

CORRECTION_MARKERS = {
    'मतलब', 'यानी', 'i mean', 'matlab', 'actually',
    'sorry', 'no no', 'नहीं नहीं', 'अरे',
}


PROLONGATION_PATTERN = re.compile(
    r'(.)\1{2,}',   
    re.UNICODE
)

DEVANAGARI_PROLONGATION = re.compile(
    r'[ऽ]{2,}|([\u0900-\u097F])\1{2,}',
    re.UNICODE
)

print(f"📚 Total fillers loaded: {len(ALL_FILLERS)}")
print(f"   Hindi: {len(HINDI_FILLERS)} | English: {len(ENGLISH_FILLERS)}")
print(f"🔧 Correction markers: {len(CORRECTION_MARKERS)}")

In [ ]:
def detect_fillers(tokens):
    """
    Detect filler words by dictionary lookup.
    Returns list of (index, matched_text).
    """
    results = []
    for i, token in enumerate(tokens):
        if token in ALL_FILLERS:
            results.append((i, token))

        if i < len(tokens) - 1:
            bigram = f"{token} {tokens[i+1]}"
            if bigram in ALL_FILLERS:
                results.append((i, bigram))
    return results


def detect_repetitions(tokens):
    """
    Detect consecutive duplicate tokens: word[i] == word[i+1].
    Returns list of (index, matched_text).
    """
    results = []
    i = 0
    while i < len(tokens) - 1:
        if tokens[i] == tokens[i + 1] and len(tokens[i]) > 1:
            
            j = i + 1
            while j < len(tokens) and tokens[j] == tokens[i]:
                j += 1
            repeated = ' '.join(tokens[i:j])
            results.append((i, repeated))
            i = j  
        else:
            i += 1
    return results


def detect_false_starts(tokens):
    """
    Detect false starts: word(s) followed by a correction marker, then different word(s).
    Pattern: token_A + correction_marker + token_B (where A != B).
    Returns list of (index, matched_text).
    """
    results = []
    for i in range(len(tokens) - 2):
        
        if tokens[i + 1] in CORRECTION_MARKERS:
            if tokens[i] != tokens[i + 2]:  
                matched = f"{tokens[i]} {tokens[i+1]} {tokens[i+2]}"
                results.append((i, matched))
        
        if i < len(tokens) - 3:
            bigram = f"{tokens[i+1]} {tokens[i+2]}"
            if bigram in CORRECTION_MARKERS:
                if tokens[i] != tokens[i + 3] if i + 3 < len(tokens) else True:
                    matched = f"{tokens[i]} {bigram} {tokens[i+3] if i+3 < len(tokens) else ''}"
                    results.append((i, matched.strip()))
    return results


def detect_prolongations(text):
    """
    Detect prolonged characters: 3+ repeated chars like आआआ, सोऽऽऽ, soooo.
    Returns list of (position, matched_text).
    """
    results = []
    
    for match in PROLONGATION_PATTERN.finditer(text):
        results.append((match.start(), match.group()))
    
    for match in DEVANAGARI_PROLONGATION.finditer(text):
        if match.group() not in [r.group() for r in PROLONGATION_PATTERN.finditer(text)]:
            results.append((match.start(), match.group()))
    return results


def detect_hesitations(segment_duration, text):
    """
    Detect hesitations based on segment duration anomalies.
    - Very short segments (< 0.5s) with few words
    - Very long segments with few words (long pauses)
    Returns True/False.
    """
    if segment_duration is None or segment_duration <= 0:
        return False
    
    word_count = len(text.split()) if text else 0
    
    
    if segment_duration < 0.5 and word_count <= 2:
        return True
    
    
    if word_count > 0:
        words_per_second = word_count / segment_duration
        if words_per_second < 0.5:  
            return True
    
    return False


print("✅ Disfluency detection functions defined.")

In [ ]:
def detect_all_disfluencies(text, segment_duration=None):
    """
    Run all disfluency detectors on a text segment.
    
    Returns:
        list of dict: [{type, matched_text}, ...]
    """
    if not text or not isinstance(text, str):
        return []
    
    processed = preprocess_text(text)
    tokens = processed.split()
    disfluencies = []
    
    
    for idx, matched in detect_fillers(tokens):
        disfluencies.append({
            'type': 'filler',
            'matched_text': matched
        })
    
    
    for idx, matched in detect_repetitions(tokens):
        disfluencies.append({
            'type': 'repetition',
            'matched_text': matched
        })
    
    
    for idx, matched in detect_false_starts(tokens):
        disfluencies.append({
            'type': 'false_start',
            'matched_text': matched
        })
    
    
    for pos, matched in detect_prolongations(processed):
        disfluencies.append({
            'type': 'prolongation',
            'matched_text': matched
        })
    
    
    if detect_hesitations(segment_duration, processed):
        disfluencies.append({
            'type': 'hesitation',
            'matched_text': f'[duration={segment_duration:.2f}s]' if segment_duration else '[pause]'
        })
    
    
    seen = set()
    unique = []
    for d in disfluencies:
        key = (d['type'], d['matched_text'])
        if key not in seen:
            seen.add(key)
            unique.append(d)
    
    return unique



demo_cases = [
    ("मैं मैं वो वो मतलब कल गया", 2.0),
    ("uh umm so I was like you know going", 3.0),
    ("आआआ सोऽऽऽ मतलब", 1.5),
    ("कल मतलब परसों गया था", 2.0),
    ("हाँ", 0.3),  
]

print("─── Detection Demo ───")
for text, dur in demo_cases:
    results = detect_all_disfluencies(text, dur)
    print(f"\n  Text: \"{text}\" (duration={dur}s)")
    if results:
        for r in results:
            print(f"    → [{r['type']}] \"{r['matched_text']}\"")
    else:
        print("    → No disfluencies detected")

## 6. Download Transcriptions & Run Segment-Level Analysis

For each recording:
1. Download the transcription JSON (with caching)
2. Parse timestamped segments
3. Run disfluency detection on each segment

In [ ]:
def download_json(url, cache_dir=CACHE_DIR):
    """
    Download a JSON file with local caching.
    Returns parsed JSON or None on failure.
    """
    if pd.isna(url) or not isinstance(url, str):
        return None
    
    
    safe_name = re.sub(r'[^\w.]', '_', url.split('/')[-1])
    cache_path = os.path.join(cache_dir, safe_name)
    
    
    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                return json.load(f)
        except:
            pass
    
    
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        data = resp.json()
        
        
        with open(cache_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False)
        
        return data
    except Exception as e:
        logger.warning(f"Failed to download {url}: {e}")
        return None


def parse_segments(transcription_data):
    """
    Parse transcription JSON into a list of segments.
    Handles various JSON structures.
    Returns list of dicts: [{text, start, end, segment_id}, ...]
    """
    segments = []
    
    if isinstance(transcription_data, list):
        items = transcription_data
    elif isinstance(transcription_data, dict):
        
        for key in ['segments', 'results', 'transcription', 'data', 'utterances']:
            if key in transcription_data:
                items = transcription_data[key]
                break
        else:
            items = [transcription_data]  
    else:
        return segments
    
    for idx, item in enumerate(items):
        if not isinstance(item, dict):
            continue
        
        
        text = None
        for key in ['text', 'transcript', 'transcription', 'sentence', 'content']:
            if key in item:
                text = str(item[key])
                break
        
        
        start = item.get('start', item.get('start_time', item.get('from', 0)))
        end = item.get('end', item.get('end_time', item.get('to', 0)))
        
        
        try:
            start = float(start)
            end = float(end)
        except (ValueError, TypeError):
            start, end = 0, 0
        
        segments.append({
            'segment_id': idx,
            'text': text or '',
            'start': start,
            'end': end,
            'duration': end - start if end > start else 0
        })
    
    return segments

print("✅ Download & parsing functions defined.")

In [ ]:


all_results = []  
stats = Counter()  
recordings_processed = 0
recordings_failed = 0

print("🔍 Starting disfluency analysis...\n")

for row_idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing recordings"):
    recording_id = row[id_col] if id_col else row_idx
    trans_url = row[trans_col] if trans_col else None
    
    
    trans_data = download_json(trans_url)
    if trans_data is None:
        recordings_failed += 1
        logger.warning(f"Recording {recording_id}: transcription download failed")
        continue
    
    segments = parse_segments(trans_data)
    if not segments:
        recordings_failed += 1
        logger.warning(f"Recording {recording_id}: no segments found")
        continue
    
    recordings_processed += 1
    
    for seg in segments:
        disfluencies = detect_all_disfluencies(
            seg['text'],
            segment_duration=seg['duration']
        )
        
        for d in disfluencies:
            stats[d['type']] += 1
            all_results.append({
                'recording_id': recording_id,
                'segment_id': seg['segment_id'],
                'start_time': seg['start'],
                'end_time': seg['end'],
                'duration': seg['duration'],
                'original_text': seg['text'],
                'disfluency_type': d['type'],
                'disfluency_text': d['matched_text'],
                'audio_clip_path': ''  
            })

print(f"\n{'='*50}")
print(f"📊 Analysis Complete!")
print(f"   Recordings processed: {recordings_processed}")
print(f"   Recordings failed   : {recordings_failed}")
print(f"   Total disfluencies  : {len(all_results)}")
print(f"\n📈 Disfluency Breakdown:")
for dtype, count in stats.most_common():
    print(f"   {dtype:15s}: {count:5d}")

## 7. Audio Clipping

For each disfluent segment:
1. Download the full audio file (lazily, with caching)
2. Clip `audio[start_time : end_time]`
3. Save as `{recording_id}_{segment_id}_{disfluency_type}.wav` (16 kHz mono)

In [ ]:
def download_audio(url, cache_dir=CACHE_DIR):
    """
    Download audio file with local caching.
    Returns local file path or None on failure.
    """
    if pd.isna(url) or not isinstance(url, str):
        return None
    
    ext = os.path.splitext(url.split('?')[0])[-1] or '.wav'
    safe_name = re.sub(r'[^\w.]', '_', url.split('/')[-1].split('?')[0])
    if not safe_name.endswith(ext):
        safe_name += ext
    cache_path = os.path.join(cache_dir, safe_name)
    
    if os.path.exists(cache_path):
        return cache_path
    
    try:
        resp = requests.get(url, timeout=120, stream=True)
        resp.raise_for_status()
        with open(cache_path, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)
        return cache_path
    except Exception as e:
        logger.warning(f"Failed to download audio {url}: {e}")
        return None


def clip_audio_pydub(audio_path, start_sec, end_sec, output_path, sample_rate=SAMPLE_RATE):
    """
    Clip audio segment using pydub. Exports as mono WAV at target sample rate.
    """
    from pydub import AudioSegment
    
    audio = AudioSegment.from_file(audio_path)

    audio = audio.set_channels(1).set_frame_rate(sample_rate)
    
    start_ms = int(start_sec * 1000)
    end_ms = int(end_sec * 1000)
    clip = audio[start_ms:end_ms]
    
    clip.export(output_path, format='wav')
    return True


def clip_audio_librosa(audio_path, start_sec, end_sec, output_path, sample_rate=SAMPLE_RATE):
    """
    Fallback: Clip audio segment using librosa + soundfile.
    """
    import librosa
    import soundfile as sf
    
    y, sr = librosa.load(audio_path, sr=sample_rate, mono=True,
                         offset=start_sec, duration=end_sec - start_sec)
    sf.write(output_path, y, sample_rate)
    return True


def clip_audio(audio_path, start_sec, end_sec, output_path):
    """
    Clip audio using pydub (primary) or librosa (fallback).
    """
    try:
        return clip_audio_pydub(audio_path, start_sec, end_sec, output_path)
    except Exception as e1:
        try:
            return clip_audio_librosa(audio_path, start_sec, end_sec, output_path)
        except Exception as e2:
            logger.warning(f"Audio clipping failed for {audio_path}: pydub={e1}, librosa={e2}")
            return False

print("✅ Audio clipping functions defined.")

In [ ]:

audio_url_map = {}
if audio_col and id_col:
    for _, row in df.iterrows():
        rid = row[id_col]
        aurl = row[audio_col]
        if pd.notna(aurl):
            audio_url_map[rid] = aurl

results_by_recording = defaultdict(list)
for i, r in enumerate(all_results):
    results_by_recording[r['recording_id']].append(i)

clips_created = 0
clips_failed = 0
audio_cache = {}  

print(f"🎬 Clipping audio for {len(results_by_recording)} recordings...\n")

for rec_id in tqdm(results_by_recording, desc="Clipping audio"):
    
    audio_url = audio_url_map.get(rec_id)
    if not audio_url:
        logger.warning(f"No audio URL for recording {rec_id}")
        clips_failed += len(results_by_recording[rec_id])
        continue
    
    if rec_id not in audio_cache:
        local_path = download_audio(audio_url)
        audio_cache[rec_id] = local_path
    local_path = audio_cache[rec_id]
    
    if not local_path:
        clips_failed += len(results_by_recording[rec_id])
        continue
    

    for idx in results_by_recording[rec_id]:
        r = all_results[idx]
        clip_name = f"{rec_id}_{r['segment_id']}_{r['disfluency_type']}.wav"
        clip_path = os.path.join(CLIPS_DIR, clip_name)
        
        success = clip_audio(
            local_path,
            r['start_time'],
            r['end_time'],
            clip_path
        )
        
        if success:
            all_results[idx]['audio_clip_path'] = clip_path
            clips_created += 1
        else:
            clips_failed += 1

print(f"\n{'='*50}")
print(f"🔊 Audio Clipping Complete!")
print(f"   Clips created: {clips_created}")
print(f"   Clips failed : {clips_failed}")

## 8. Create Structured CSV Dataset

Output schema: `recording_id | segment_id | disfluency_type | disfluency_text | audio_clip_path`

In [ ]:

results_df = pd.DataFrame(all_results)

column_order = [
    'recording_id', 'segment_id', 'disfluency_type', 'disfluency_text',
    'original_text', 'start_time', 'end_time', 'duration', 'audio_clip_path'
]
available_cols = [c for c in column_order if c in results_df.columns]
results_df = results_df[available_cols]

before_dedup = len(results_df)
results_df = results_df.drop_duplicates(
    subset=['recording_id', 'segment_id', 'disfluency_type', 'disfluency_text']
)
after_dedup = len(results_df)

results_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"💾 CSV saved to: {OUTPUT_CSV}")
print(f"   Total rows: {after_dedup} (removed {before_dedup - after_dedup} duplicates)")
print(f"\n📋 CSV Schema:")
print(f"   {list(results_df.columns)}")
print(f"\n🔎 Preview:")
results_df.head(10)

## 9. Quality Control & Reporting

- Random 5–10% sample verification
- Per-type and per-recording summary
- Flagging potential false positives

In [ ]:
print("="*60)
print("            📊 QUALITY CONTROL REPORT")
print("="*60)

print(f"\n🔢 Overall Statistics:")
print(f"   Total recordings in dataset  : {len(df)}")
print(f"   Recordings processed         : {recordings_processed}")
print(f"   Recordings failed            : {recordings_failed}")
print(f"   Total disfluencies detected  : {len(results_df)}")
print(f"   Unique recordings with disfl.: {results_df['recording_id'].nunique()}")

print(f"\n📈 Disfluency Type Distribution:")
type_counts = results_df['disfluency_type'].value_counts()
for dtype, count in type_counts.items():
    pct = count / len(results_df) * 100
    bar = '█' * int(pct / 2)
    print(f"   {dtype:15s}: {count:5d} ({pct:5.1f}%) {bar}")

print(f"\n📋 Top 10 Recordings by Disfluency Count:")
rec_counts = results_df.groupby('recording_id').size().sort_values(ascending=False).head(10)
for rec_id, count in rec_counts.items():
    print(f"   Recording {rec_id}: {count} disfluencies")

avg_per_rec = results_df.groupby('recording_id').size().mean()
print(f"\n📊 Average disfluencies per recording: {avg_per_rec:.1f}")

In [ ]:
sample_pct = 0.07  
sample_size = max(1, int(len(results_df) * sample_pct))
sample_indices = random.sample(range(len(results_df)), min(sample_size, len(results_df)))
sample_df = results_df.iloc[sample_indices]

print(f"\n{'='*60}")
print(f"🔍 RANDOM SAMPLE FOR MANUAL VERIFICATION")
print(f"   Sample size: {len(sample_df)} / {len(results_df)} ({sample_pct*100:.0f}%)")
print(f"{'='*60}\n")

for i, (_, row) in enumerate(sample_df.iterrows()):
    print(f"  Sample #{i+1}:")
    print(f"    Recording  : {row['recording_id']}")
    print(f"    Segment    : {row['segment_id']}")
    print(f"    Type       : {row['disfluency_type']}")
    print(f"    Matched    : \"{row['disfluency_text']}\"")
    if 'original_text' in row:
        print(f"    Full text  : \"{row['original_text'][:80]}...\"" if len(str(row['original_text'])) > 80 else f"    Full text  : \"{row['original_text']}\"")
    if row.get('audio_clip_path'):
        exists = os.path.exists(row['audio_clip_path'])
        print(f"    Clip exists: {'✅' if exists else '❌'} {row['audio_clip_path']}")
    print()

In [ ]:

print("\n" + "="*60)
print("            ✅ PIPELINE COMPLETE")
print("="*60)
print(f"\n📁 Output Files:")
print(f"   📄 Results CSV   : {OUTPUT_CSV}")
print(f"   📂 Audio clips   : {CLIPS_DIR}/")
print(f"   📝 Error log     : {ERROR_LOG}")
print(f"   💾 Cache dir     : {CACHE_DIR}/")

clip_files = [f for f in os.listdir(CLIPS_DIR) if f.endswith('.wav')] if os.path.exists(CLIPS_DIR) else []
print(f"\n🔊 Total audio clips on disk: {len(clip_files)}")
print(f"📊 Total disfluency records : {len(results_df)}")

print(f"\n🎯 Next Steps:")
print(f"   1. Review the random sample above for accuracy")
print(f"   2. Listen to a few audio clips to verify disfluency presence")
print(f"   3. Check {ERROR_LOG} for any download/processing failures")
print(f"   4. Open {OUTPUT_CSV} in Excel for full results")